Part 1 — Dataset preparation

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
    session_id INT,
    zone_id STRING,
    vehicle_type STRING,
    duration_hours INT,
    parking_fee INT,
    status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
    zone_id STRING,
    zone_name STRING,
    city STRING,
    capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
);

Part 2 — Assessment answers

In [0]:
%sql
select session_id, vehicle_type, parking_fee from parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
select * from parking_sessions_source where status = 'Completed' and parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
select vehicle_type,sum(parking_fee) as total_parking_fee, count(*) as completed_sessions_count from parking_sessions_source where status = 'Completed' group by vehicle_type;

vehicle_type,total_parking_fee,completed_sessions_count
Car,1750,8
Bike,160,4
SUV,540,2


In [0]:
from pyspark.sql.functions import col, when
df = spark.table("parking_sessions_source")
df_with_category = df.withColumn(
    "fee_category",when(col("parking_fee") >= 200, "High").otherwise("Low")
)
display(df_with_category)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


Section B

In [0]:
%sql
create or replace table parking_summary using delta as select * from parking_sessions_work where status = 'Completed';

num_affected_rows,num_inserted_rows


In [0]:
%sql
insert into parking_sessions_work values (123, 'Z01', 'Car', 3, 150, 'Active');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
update parking_sessions_work set status = 'Completed',parking_fee = 280 where session_id = 104;

num_affected_rows
1


In [0]:
%sql
delete from parking_sessions_work where status = 'Cancelled';
select * from parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed


Section C

In [0]:
%sql
select * from incoming_parking_sessions inc left join parking_sessions_work work on inc.session_id = work.session_id is not null;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed,104,Z01,SUV,4,280,Completed
102,Z02,Bike,3,60,Completed,101,Z01,Car,3,150,Completed
102,Z02,Bike,3,60,Completed,102,Z02,Bike,2,40,Completed
102,Z02,Bike,3,60,Completed,103,Z03,Car,5,250,Completed
102,Z02,Bike,3,60,Completed,105,Z04,Bike,1,20,Completed
102,Z02,Bike,3,60,Completed,106,Z02,Car,6,300,Completed
102,Z02,Bike,3,60,Completed,108,Z05,Car,3,150,Completed
102,Z02,Bike,3,60,Completed,109,Z01,Bike,4,80,Completed
102,Z02,Bike,3,60,Completed,110,Z04,SUV,5,300,Active
102,Z02,Bike,3,60,Completed,111,Z02,Car,2,100,Completed


In [0]:
%sql
merge into parking_sessions_work as target using incoming_parking_sessions as source on target.session_id = source.session_id
when matched then 
update set *
when not matched then
 insert (session_id, zone_id, vehicle_type, duration_hours, parking_fee, status)
values (source.session_id, source.zone_id, source.vehicle_type, source.duration_hours, source.parking_fee, source.status);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T08:59:56.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1637634183392900),af6603d9-c40d-457d-aeda-2a6f0514bac8,1106-083731-nh6xijcv-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9827, p25FileSize -> 2296, numDeletionVectorsRemoved -> 1, minFileSize -> 2296, numAddedFiles -> 1, maxFileSize -> 2296, p75FileSize -> 2296, p50FileSize -> 2296, numAddedBytes -> 2296)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T08:59:55.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(session_id#13566 = session_id#13548)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1637634183392900),af6603d9-c40d-457d-aeda-2a6f0514bac8,1106-083731-nh6xijcv-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7534, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3858, materializeSourceTimeMs -> 324, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1257, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2202)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T08:47:05.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1637634183392900),15bbbcde-fa6a-4c66-8bd6-e3bb46602d1e,1106-083731-nh6xijcv-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6105, p25FileSize -> 2293, numDeletionVectorsRemoved -> 1, minFileSize -> 2293, numAddedFiles -> 1, maxFileSize -> 2293, p75FileSize -> 2293, p50FileSize -> 2293, numAddedBytes -> 2293)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T08:47:02.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#12621 = Cancelled)""])",null,List(1637634183392900),15bbbcde-fa6a-4c66-8bd6-e3bb46602d1e,1106-083731-nh6xijcv-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1371, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 858, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 512)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T08:46:16.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#12117 = 104)""])",null,List(1637634183392900),5d391a39-4340-4247-a545-eea282f2bc25,1106-083731-nh6xijcv-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3855, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1823, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1922, rewriteTimeMs -> 2004)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T08:45:43.000Z,142253823518303,azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Ap

In [0]:
%sql
select * from parking_sessions_work version as of 1 where session_id = 102;
select * from parking_sessions_work where session_id = 102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


Section D

In [0]:
%sql
create or replace view completed_parking_view as
select * from parking_sessions_work where status = 'Completed';

In [0]:
%sql
create or replace temp view temp_zone_z01 as
select * from parking_sessions_work where zone_id = 'Z01';

create or replace global temp view gtv_high_fees as
select * from parking_sessions_work where parking_fee > 200;
select * from temp_zone_z01;
select * from global_temp.gtv_high_fees;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7339878997414715>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create or replace temp view temp_zone_z01 as\nselect * from parking_sessions_work where zone_id = 'Z01';\n\ncreate or replace global temp view gtv_high_fees as\nselect * from parking_sessions_work where parking_fee > 200;\nselect * from temp_zone_z01;\nselect * from global_temp.gtv_high_fees;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is 

i coudnt get the results for the above query as im using serverless connection

In [0]:
%sql
select pz.zone_name,pz.city,psw.vehicle_type,psw.parking_fee from parking_sessions_work psw join parking_zones pz on psw.zone_id = pz.zone_id;

zone_name,city,vehicle_type,parking_fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
%sql
describe detail parking_sessions_work;
optimize parking_sessions_work;
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a943bf1f-3b3f-4f62-af3d-06eabf666065,dbt.default.parking_sessions_work,null,abfss://unity-catalog-storage@dbstorageikg7sh2ow3os2.dfs.core.windows.net/7405618747972166/__unitystorage/catalogs/567c7b20-d0e0-497e-b6c5-9ad5d4ac644e/tables/ea745ec1-3061-4664-a78f-081b578e650b,2026-11-06T08:38:47.632Z,2026-11-06T08:59:56.000Z,List(),List(),1,2296,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Section E

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbt.default.my_checkpoints;

-- dbt is my workspace name

In [0]:

spark.sql("CREATE VOLUME IF NOT EXISTS dbt.default.streaming_checkpoints")

rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

query = rate_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/dbt/default/streaming_checkpoints/rate_console") \
    .trigger(availableNow=True) \
    .start()

In [0]:
from pyspark.sql.functions import col

transformed_df = rate_df \
    .withColumn("amount", col("value") * 100) \
    .filter(col("amount") > 300)

query = transformed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/dbt/default/streaming_checkpoints/rate_transformed_console") \
    .trigger(availableNow=True) \
    .start()

In [0]:
from pyspark.sql.functions import col

transformed_df = rate_df \
    .withColumn("amount", col("value") * 100) \
    .filter(col("amount") > 300)

# Process batch once and stop
query = transformed_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/dbt/default/streaming_checkpoints/rate_transformed_temp") \
    .trigger(availableNow=True) \
    .table("dbt.default.rate_transformed_temp")

query.awaitTermination()
display(spark.read.table("dbt.default.rate_transformed_temp"))

timestamp,value,amount
2026-11-06T10:33:45.555Z,1399,139900
2026-11-06T10:33:47.155Z,1407,140700
2026-11-06T10:33:48.755Z,1415,141500
2026-11-06T10:33:50.355Z,1423,142300
2026-11-06T10:33:51.955Z,1431,143100
2026-11-06T10:33:53.555Z,1439,143900
2026-11-06T10:33:55.155Z,1447,144700
2026-11-06T10:33:56.755Z,1455,145500
2026-11-06T10:33:58.355Z,1463,146300
2026-11-06T10:33:59.955Z,1471,147100


In [0]:
%sql
create or refresh streaming table bronze_parking_sessions as
select * from stream(LIVE.parking_sessions_source);

create or refresh streaming table silver_parking_sessions as
select * from stream(LIVE.bronze_parking_sessions)
where status = 'Completed';

message
The operation was successfully executed.


In [0]:
%sql
create or refresh materialized view gold_zone_summary as 
select zone_id, count(*) as total_completed_sessions, sum(parking_fee) as total_parking_revenue from LIVE.silver_parking_sessions group by zone_id;

result
The operation was successfully executed.
